# Project 3: MediaPipe Gesture Recognition

Course lesson: [P3.2 The MediaPipe Colab Notebook](https://neel429.github.io/ml-for-robotics-/index.html?lesson=proj3-mediapipe-colab)

This notebook teaches MediaPipe hand tracking on a still image before you run the live webcam test in VS Code and connect gestures to the robot.

## Cell 1: Install and import

MediaPipe is not pre-installed in Colab. The first cell installs it. The `-q` flag suppresses the long install output.

In [ ]:
!pip install mediapipe -q

import cv2
import mediapipe as mp
import numpy as np
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

print("MediaPipe version:", mp.__version__)

## Cell 2: Initialize the hands detector

`static_image_mode=False` means video mode. MediaPipe uses temporal information between frames to track hands once detected, which is faster than detecting from scratch every frame.

`max_num_hands=2` is required because the robot gesture logic only moves when both hands are visible.

`min_detection_confidence=0.7` only reports a detection when the model is at least 70% confident.

`min_tracking_confidence=0.6` keeps tracking an already detected hand as long as tracking confidence stays above 60%.

Some newer MediaPipe builds do not expose `mp.solutions` at the top level. The `try/except` block below keeps the same Hands API working in both older and newer installs.

In [ ]:
try:
    mp_hands = mp.solutions.hands
    mp_draw  = mp.solutions.drawing_utils
except AttributeError:
    from mediapipe.python.solutions import hands as mp_hands
    from mediapipe.python.solutions import drawing_utils as mp_draw


hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6
)

print("Hand detector initialized.")
print(f"Watching for up to {hands.max_num_hands} hands")

## Cell 3: Test on a still image

Upload a photo of your own hand to Colab with the Files panel or the upload prompt below. A photo named `hand_test.jpg` is fine, but the cell uses whatever file you upload first.

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload your hand photo

img      = cv2.imread(list(uploaded.keys())[0])
img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
results  = hands.process(img_rgb)

print(f"Hands detected: {len(results.multi_hand_landmarks) if results.multi_hand_landmarks else 0}")

if results.multi_hand_landmarks:
    for i, hand_lm in enumerate(results.multi_hand_landmarks):
        mp_draw.draw_landmarks(img_rgb, hand_lm, mp_hands.HAND_CONNECTIONS)
        print(f"\nHand {i+1} - first 5 landmark coordinates:")
        for j in range(5):
            lm = hand_lm.landmark[j]
            print(f"  Landmark {j}: x={lm.x:.3f}  y={lm.y:.3f}  z={lm.z:.3f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title("MediaPipe hand landmarks")
plt.axis("off")
plt.show()

`hands.process()` requires RGB input. OpenCV loads images in BGR order, so `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` runs before MediaPipe. `results.multi_hand_landmarks` is a list of detected hands, and each hand has 21 normalized landmark coordinates.

## Cell 4: Read specific landmarks

The core comparison is `tip.y < pip.y`: lower y means higher in the image, because y=0 is the top and y=1 is the bottom.

In [ ]:
if results.multi_hand_landmarks:
    hand = results.multi_hand_landmarks[0]
    lm   = hand.landmark

    print("Key landmarks for finger detection:")
    print(f"  Wrist (0):           x={lm[0].x:.3f}  y={lm[0].y:.3f}")
    print(f"  Index tip (8):       x={lm[8].x:.3f}  y={lm[8].y:.3f}")
    print(f"  Index PIP (6):       x={lm[6].x:.3f}  y={lm[6].y:.3f}")
    print()

    index_extended = lm[8].y < lm[6].y
    print(f"Index finger extended: {index_extended}")
    print(f"  tip.y={lm[8].y:.3f}  pip.y={lm[6].y:.3f}")
    print(f"  tip is {'ABOVE' if index_extended else 'BELOW'} PIP = finger is {'extended' if index_extended else 'curled'}")

## Cell 5: The is_hand_open function

This is the same open/closed rule used by the robot control script.

In [ ]:
def is_hand_open(hand_landmarks):
    pts    = hand_landmarks.landmark
    count  = 0

    # Thumb: compare x-distance from wrist
    # tip farther from wrist than IP joint = thumb extended
    if pts[4].x > pts[2].x:
        count += 1

    # Four fingers: tip above PIP = finger extended
    for tip, pip in zip([8, 12, 16, 20], [6, 10, 14, 18]):
        if pts[tip].y < pts[pip].y:
            count += 1

    return count >= 3   # open if 3 or more digits extended

print("Testing is_hand_open on your uploaded photo:")
if results.multi_hand_landmarks:
    result = is_hand_open(results.multi_hand_landmarks[0])
    print(f"  Hand is: {'OPEN' if result else 'CLOSED'}")

`pts[4]` is the thumb tip and `pts[2]` is the thumb MCP. The four finger pairs are `8/6`, `12/10`, `16/14`, and `20/18`. Open means at least 3 of the 5 digits are extended, which tolerates slightly bent fingers.

## Cell 6: Two-hand gesture classification

In [ ]:
def classify_gesture(hands_dict):
    if 'Left' not in hands_dict or 'Right' not in hands_dict:
        return "stop"   # need both hands

    lo = hands_dict['Left']['open']
    ro = hands_dict['Right']['open']

    if   lo and ro:         return "forward"
    elif not lo and not ro: return "backward"
    elif ro and not lo:     return "right"
    elif lo and not ro:     return "left"
    return "stop"

demo = {
    'Left':  {'open': True},
    'Right': {'open': False},
}
print(classify_gesture(demo))

| Left hand | Right hand | Command |
|---|---|---|
| OPEN | OPEN | FORWARD |
| CLOSED | CLOSED | BACKWARD |
| CLOSED | OPEN | TURN RIGHT |
| OPEN | CLOSED | TURN LEFT |
| One hand | --- | STOP |

Requiring both hands is a safety design: the robot only moves when you deliberately hold both hands in the frame.

## Cell 7: Live webcam test runs in VS Code, not Colab

Colab cannot access the laptop webcam directly in a standard cell. The `gesture_recognize.py` file in the repository runs live gesture detection on your laptop webcam using `cv2.VideoCapture(0)`. Run that file locally in VS Code after this notebook.

The notebook teaches the concepts on still images. The live test runs locally. This is the same split as the lane follower project: understand the algorithm in Colab, apply it in VS Code.